In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import plotly.graph_objects as go
import math

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

def load_dataset(filename):
    BASE_DIR = './data'
    EXTENSION = '.csv'
    TARGET_PATH = f'{BASE_DIR}/{filename}{EXTENSION}'
    try:
        return pd.read_csv(TARGET_PATH)
    except FileNotFoundError:
        return None
    except UnicodeDecodeError:
        return pd.read_csv(TARGET_PATH, encoding='euc-kr')

df = load_dataset('크루즈입항')

In [4]:
import pandas as pd
import plotly.graph_objects as go

# ============================================================
# 1. 색상 설정
# ============================================================

BLUE = "#3478C0"
ORANGE = "#E67E22"
GRAY = "#657487"


# ============================================================
# 2. 데이터 전처리
# ============================================================

df_month = df.copy()

df_month['입항일시'] = pd.to_datetime(
    df_month['입항일시'], errors='coerce'
)

df_month = df_month.dropna(subset=['입항일시']).copy()

df_month = df_month[
    (df_month['입출'].astype(str).str.strip() == '입항') &
    (df_month['선박용도'].astype(str).str.strip() == '크루즈선')
].copy()


# ============================================================
# 3. 코로나 전후 기간
# ============================================================

before = df_month[
    (df_month['입항일시'] >= '2013-01-01') &
    (df_month['입항일시'] < '2020-01-01')
].copy()

after = df_month[
    (df_month['입항일시'] >= '2023-01-01') &
    (df_month['입항일시'] < '2026-10-17')
].copy()

before['연도'] = before['입항일시'].dt.year
before['월'] = before['입항일시'].dt.month

after['연도'] = after['입항일시'].dt.year
after['월'] = after['입항일시'].dt.month


# ============================================================
# 4. 연도별 월 입항건수
# ============================================================

before_monthly = (
    before.groupby(['연도', '월'])
    .size()
    .unstack()
    .reindex(index=range(2013, 2020), columns=range(1, 13))
)

after_monthly = (
    after.groupby(['연도', '월'])
    .size()
    .unstack()
    .reindex(index=range(2023, 2027), columns=range(1, 13))
)


# ============================================================
# 5. 월별 평균
# ============================================================

before_avg = before_monthly.mean(axis=0)
after_avg = after_monthly.mean(axis=0)

months = [f'{m}월' for m in range(1, 13)]


# ============================================================
# 6. 선그래프 생성 (기존 유지)
# ============================================================

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=months,
    y=before_avg.values,
    mode='lines+markers',
    name='코로나 이전 · 2013~2019년',

    line=dict(color=BLUE, width=3),

    marker=dict(
        size=8,
        color=BLUE,
        line=dict(color='#245A91', width=1.5)
    ),

    hovertemplate=(
        '%{x}<br>'
        '월평균 입항건수: %{y:.1f}건'
        '<extra></extra>'
    )
))

fig.add_trace(go.Scatter(
    x=months,
    y=after_avg.values,
    mode='lines+markers',
    name='코로나 이후 · 2023~2026.10.16',

    line=dict(color=ORANGE, width=3),

    marker=dict(
        size=8,
        color=ORANGE,
        line=dict(color='#B85C0A', width=1.5)
    ),

    hovertemplate=(
        '%{x}<br>'
        '월평균 입항건수: %{y:.1f}건'
        '<extra></extra>'
    )
))


# ============================================================
# 7. 하단 주석
# ============================================================

note = (
    '※ 2020~2022년 제외. 월별 평균 입항건수 비교.<br>'
    '※ 10월은 2026년 16일까지 집계, 11~12월은 2023~2025년 기준.<br>'
    '※ 기록이 없는 연월은 평균 계산에서 제외.'
)


# ============================================================
# 8. 그래프 디자인 (제목·부제목 중앙정렬)
# ============================================================

fig.update_layout(
    title=dict(
        text=(
            '<b>부산항 크루즈 코로나 전후 월별 평균 입항건수 비교</b>'
            '<br>'
            f"<span style='font-size:18px;color:{GRAY}'>"
            '월별 입항 추이 분석'
            '</span>'
        ),

        # 제목 및 부제목 중앙정렬
        x=0.5,
        y=0.95,
        xanchor='center',
        yanchor='top',

        font=dict(
            family='Malgun Gothic, Arial, sans-serif',
            size=23,
            color='#243247'
        )
    ),

    template='plotly_white',

    width=1200,
    height=675,

    paper_bgcolor='white',
    plot_bgcolor='white',

    font=dict(
        family='Malgun Gothic, Arial, sans-serif',
        size=18,
        color='#243247'
    ),

    # X축
    xaxis=dict(
        title=None,
        type='category',
        categoryorder='array',
        categoryarray=months,
        showgrid=False,
        showline=False,
        tickfont=dict(size=17, color='#243247'),
        fixedrange=True
    ),

    # Y축
    yaxis=dict(
        title='월평균 입항건수 (건)',
        rangemode='tozero',
        showgrid=True,
        gridcolor='#E9EDF2',
        zeroline=False,
        showline=False,
        tickfont=dict(size=16, color='#243247'),
        fixedrange=True
    ),

    # 범례 중앙정렬
    legend=dict(
        orientation='h',
        y=1.08,
        x=0.5,
        xanchor='center',
        font=dict(size=14, color='#243247')
    ),

    margin=dict(
        l=90,
        r=45,
        t=125,
        b=125
    ),

    hovermode='x unified',
    showlegend=True,

    # 외곽 테두리
    shapes=[
        dict(
            type='rect',
            xref='paper',
            yref='paper',
            x0=0,
            y0=0,
            x1=1,
            y1=1,
            line=dict(color='#E9EDF2', width=1),
            fillcolor='rgba(0,0,0,0)',
            layer='above'
        )
    ],

    # 하단 주석
    annotations=[
        dict(
            xref='paper',
            yref='paper',
            x=0,
            y=-0.20,
            text=note,
            showarrow=False,
            xanchor='left',
            align='left',

            font=dict(
                family='Malgun Gothic, Arial, sans-serif',
                size=14,
                color=GRAY
            )
        )
    ]
)


# ============================================================
# 9. 그래프 출력
# ============================================================

fig.show(config={
    'displaylogo': False,

    'toImageButtonOptions': {
        'format': 'png',
        'filename': 'PPT_코로나전후_월별평균입항건수',
        'width': 1200,
        'height': 675,
        'scale': 3
    }
})